# EDA — PaySim Financial Fraud Detection
**Project:** Financial Transaction Fraud Detection System using Big Data Analytics  
**Member 2 Task:** Exploratory Data Analysis (EDA)  
**Dataset:** PaySim Synthetic Financial Dataset (Kaggle)  
**File:** `eda_paysim.ipynb`

---
This notebook explores the PaySim dataset to understand patterns, class imbalance, and feature relationships before building the ML model.

In [ ]:
# ============================================================
# CELL 1 — Import Libraries
# ============================================================

# pandas  : for loading and manipulating tabular data (like Excel, but in Python)
# numpy   : for math operations on arrays/numbers
# matplotlib : for creating charts and graphs
# seaborn : a prettier wrapper on top of matplotlib for statistical plots

import pandas as pd        # 'pd' is just a short nickname we give pandas
import numpy as np         # 'np' is the standard nickname for numpy
import matplotlib.pyplot as plt  # 'plt' is the standard nickname
import seaborn as sns      # 'sns' is the standard nickname for seaborn
import os                  # for creating folders if they don't exist
import warnings            # to suppress non-critical warnings

# ---- Configuration ----
# Change this variable if your CSV file is in a different location
DATA_PATH = 'paysim.csv'

# This folder will store all the plot PNG files we generate
PLOT_DIR = 'eda_plots'
os.makedirs(PLOT_DIR, exist_ok=True)  # Create folder only if it doesn't already exist

# Make all seaborn plots look clean and modern
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)

# Hide harmless warnings so output stays clean
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')
print(f'   Plots will be saved to: ./{PLOT_DIR}/')

**What Cell 1 does:**  
Imports all the Python packages we need and sets up a folder (`eda_plots/`) where all our chart images will be saved. Think of `import` like `#include` in C++ — it loads external libraries into our program.

In [ ]:
# ============================================================
# CELL 2 — Load Dataset & Basic Info
# ============================================================

# pd.read_csv() reads the CSV file into a 'DataFrame' — like a table in memory
print(f'Loading dataset from: {DATA_PATH} ...')
df = pd.read_csv(DATA_PATH)

print('\n--- Dataset Shape ---')
# .shape returns (number_of_rows, number_of_columns) — like a 2D array size
print(f'  Rows    : {df.shape[0]:,}')   # {:,} adds comma separators for large numbers
print(f'  Columns : {df.shape[1]}')

print('\n--- Column Names ---')
# .columns gives us the list of column header names
for col in df.columns:
    print(f'  • {col}')

print('\n--- First 3 Rows (sample of data) ---')
# .head(3) returns the first 3 rows — helpful to see what the data looks like
df.head(3)

**What Cell 2 does:**  
Loads the entire 6.3 million row CSV file into memory as a DataFrame (think of it as a 2D array where columns have names). We print the shape so we know how big the dataset is, and peek at the first 3 rows to see what the data looks like.

In [ ]:
# ============================================================
# CELL 3 — Data Types & Null Value Check
# ============================================================

print('--- Column Data Types ---')
# .dtypes tells us the data type of each column (int64, float64, object = string, etc.)
print(df.dtypes)

print('\n--- Null (Missing) Value Counts per Column ---')
# .isnull() returns True/False for each cell — .sum() counts the Trues (nulls)
null_counts = df.isnull().sum()
print(null_counts)

# Check if there are ANY nulls at all
total_nulls = null_counts.sum()
if total_nulls == 0:
    print('\n✅ No missing values found! Dataset is clean.')
else:
    print(f'\n⚠️  Total missing values: {total_nulls}')

print('\n--- Basic Statistics (numeric columns only) ---')
# .describe() gives count, mean, std, min, max etc. — like a quick stats summary
df.describe()

**What Cell 3 does:**  
Checks the data type of every column (important for ML — models need numbers, not text). Also checks for missing/null values. In real-world datasets, missing values are common and must be handled. PaySim is clean, so we expect zero nulls.

In [ ]:
# ============================================================
# CELL 4 — Fraud vs Legit Class Distribution
# ============================================================

print('--- Fraud vs Legit Transaction Counts ---')

# Count how many rows have isFraud=0 (legit) and isFraud=1 (fraud)
# .value_counts() is like a frequency counter
class_counts = df['isFraud'].value_counts()

total_transactions = len(df)   # total number of rows
fraud_count = class_counts[1]  # how many are fraud (label=1)
legit_count  = class_counts[0] # how many are legit (label=0)

# Calculate percentages
fraud_pct = (fraud_count / total_transactions) * 100
legit_pct  = (legit_count  / total_transactions) * 100

print(f'  Total Transactions : {total_transactions:,}')
print(f'  Legitimate (0)     : {legit_count:,}  ({legit_pct:.2f}%)')
print(f'  Fraudulent  (1)    : {fraud_count:,}   ({fraud_pct:.4f}%)')
print(f'\n  ⚠️  Class Imbalance Ratio → {int(legit_count/fraud_count):,}:1  (legit:fraud)')
print('  This means for every 1 fraud, there are ~770 legitimate transactions.')
print('  This severe imbalance is why plain accuracy is USELESS here!')
print('  A model that predicts everything as \'legit\' would be 99.87% accurate — but catch 0 frauds!')

**What Cell 4 does:**  
Counts how many transactions are fraud vs. legitimate and calculates the percentage of each. This reveals the **extreme class imbalance** — only ~0.13% are fraud. This is the most important insight in the EDA and drives all our later decisions (class weights instead of accuracy metric).

In [ ]:
# ============================================================
# CELL 5 — Plot Class Imbalance (Pie Chart + Bar Chart)
# ============================================================

# We create a figure with 2 side-by-side subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Class Imbalance: Fraud vs Legitimate Transactions', fontsize=15, fontweight='bold')

labels  = ['Legitimate (0)', 'Fraud (1)']
counts  = [legit_count, fraud_count]
colors  = ['#2ecc71', '#e74c3c']  # green for legit, red for fraud
explode = [0, 0.12]               # 'explode' pops the fraud slice out slightly

# ---- Pie Chart (left subplot) ----
axes[0].pie(
    counts,
    labels=labels,
    colors=colors,
    explode=explode,
    autopct='%1.3f%%',    # show percentage with 3 decimal places
    startangle=140,
    shadow=True
)
axes[0].set_title('Proportion of Each Class')

# ---- Bar Chart (right subplot) ----
bars = axes[1].bar(labels, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[1].set_title('Count of Each Class')
axes[1].set_ylabel('Number of Transactions')
axes[1].set_yscale('log')   # log scale because fraud count is tiny vs legit
axes[1].set_xlabel('Transaction Class')

# Add count labels on top of each bar
for bar, count in zip(bars, counts):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,  # x position: center of bar
        bar.get_height() * 1.05,            # y position: just above bar
        f'{count:,}',                       # label text with comma formatting
        ha='center', va='bottom', fontweight='bold', fontsize=10
    )

plt.tight_layout()  # automatically adjusts spacing so nothing overlaps

# Save the figure to PNG file
save_path = os.path.join(PLOT_DIR, 'class_imbalance.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()  # display inline in Jupyter
print(f'✅ Plot saved to: {save_path}')

**What Cell 5 does:**  
Creates two charts side-by-side — a pie chart showing proportions and a bar chart (on log scale) showing counts. The log scale on the bar chart is necessary because the fraud bar would be nearly invisible at normal scale (0.13% vs 99.87%). Both charts are saved as a single PNG file.

In [ ]:
# ============================================================
# CELL 6 — Fraud Rate per Transaction Type
# ============================================================

print('--- Fraud Analysis by Transaction Type ---\n')

# Group all rows by the 'type' column, then for each group:
#   - Count total rows         → how many transactions of that type exist
#   - Sum the 'isFraud' column → since isFraud is 0 or 1, sum = fraud count
fraud_by_type = df.groupby('type')['isFraud'].agg(
    total_count='count',          # total transactions in this type
    fraud_count='sum'             # number of fraudulent transactions in this type
).reset_index()  # converts the grouped index back into a regular column

# Calculate fraud rate: what % of transactions in each type are fraud?
fraud_by_type['fraud_rate_pct'] = (
    fraud_by_type['fraud_count'] / fraud_by_type['total_count'] * 100
).round(4)

# Sort by fraud count (highest first)
fraud_by_type = fraud_by_type.sort_values('fraud_count', ascending=False)

print(fraud_by_type.to_string(index=False))

print('\n🔍 Key Insight:')
print('   TRANSFER and CASH_OUT are the ONLY types with fraud!')
print('   PAYMENT, DEBIT, and CASH_IN have zero fraud cases.')
print('   This is a critical domain fact — we will filter to only these two types.')

**What Cell 6 does:**  
Groups the data by transaction type (PAYMENT, TRANSFER, CASH_OUT, etc.) and counts how many frauds appear in each group. This confirms a critical domain fact: **fraud only happens in TRANSFER and CASH_OUT**. This justifies filtering out other types to reduce noise in our model.

In [ ]:
# ============================================================
# CELL 7 — Plot Fraud Count by Transaction Type
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Fraud Distribution by Transaction Type', fontsize=14, fontweight='bold')

# ---- Left: Fraud Count per Type ----
bar_colors = ['#e74c3c' if t in ['TRANSFER', 'CASH_OUT'] else '#95a5a6' 
              for t in fraud_by_type['type']]

bars = axes[0].bar(
    fraud_by_type['type'],
    fraud_by_type['fraud_count'],
    color=bar_colors,
    edgecolor='black',
    linewidth=0.7
)
axes[0].set_title('Fraud Count by Type')
axes[0].set_xlabel('Transaction Type')
axes[0].set_ylabel('Number of Fraudulent Transactions')

# Add count labels on bars
for bar in bars:
    h = bar.get_height()
    if h > 0:   # only label bars with non-zero height
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            h + 50,
            f'{int(h):,}',
            ha='center', va='bottom', fontsize=9, fontweight='bold'
        )

# ---- Right: Fraud Rate per Type ----
# Filter only types that have at least 1 fraud
fraud_types_only = fraud_by_type[fraud_by_type['fraud_count'] > 0]

axes[1].barh(
    fraud_types_only['type'],
    fraud_types_only['fraud_rate_pct'],
    color=['#e74c3c', '#e67e22'],
    edgecolor='black'
)
axes[1].set_title('Fraud Rate (%) by Type')
axes[1].set_xlabel('Fraud Rate (%)')
axes[1].set_ylabel('Transaction Type')

# Add rate labels
for i, (rate, txn_type) in enumerate(zip(fraud_types_only['fraud_rate_pct'], fraud_types_only['type'])):
    axes[1].text(rate + 0.01, i, f'{rate:.2f}%', va='center', fontweight='bold')

plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'fraud_by_type.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved to: {save_path}')

**What Cell 7 does:**  
Creates two charts — one showing the raw count of fraud per transaction type, and one showing the fraud *rate* (percentage). Red bars highlight the fraudulent types. The gray bars show types with zero fraud, confirming they can be safely filtered out.

In [ ]:
# ============================================================
# CELL 8 — Filter to TRANSFER and CASH_OUT Only
# ============================================================

print('Filtering dataset to only TRANSFER and CASH_OUT transactions...')

# .isin() checks if each value in 'type' is in the given list
# This is like: SELECT * FROM df WHERE type IN ('TRANSFER', 'CASH_OUT')
df_filtered = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
# .copy() creates an independent copy so edits don't affect the original df

print(f'\n--- Shape Comparison ---')
print(f'  Original dataset  : {df.shape[0]:,} rows')
print(f'  Filtered dataset  : {df_filtered.shape[0]:,} rows')
print(f'  Rows removed      : {df.shape[0] - df_filtered.shape[0]:,}')

# Recount fraud/legit in the filtered dataset
filtered_fraud = df_filtered['isFraud'].sum()
filtered_legit = len(df_filtered) - filtered_fraud
filtered_fraud_pct = (filtered_fraud / len(df_filtered)) * 100

print(f'\n--- Fraud in Filtered Dataset ---')
print(f'  Legitimate : {filtered_legit:,}  ({100-filtered_fraud_pct:.2f}%)')
print(f'  Fraud      : {filtered_fraud:,}    ({filtered_fraud_pct:.4f}%)')
print(f'\n✅ df_filtered created with {df_filtered.shape[0]:,} rows.')
print('   This will be used for all further analysis and model training.')

**What Cell 8 does:**  
Creates a new filtered DataFrame called `df_filtered` that contains only TRANSFER and CASH_OUT rows. Since fraud *never* occurs in other types, removing those rows reduces noise and makes the model focus on the right patterns. The `.copy()` is important — without it, modifying `df_filtered` later could accidentally modify `df` too (a common Python gotcha).

In [ ]:
# ============================================================
# CELL 9 — Amount Distribution: Fraud vs Legit (Linear + Log)
# ============================================================

# Separate fraud and legit rows for plotting
fraud_rows = df_filtered[df_filtered['isFraud'] == 1]['amount']
legit_rows = df_filtered[df_filtered['isFraud'] == 0]['amount']

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
fig.suptitle('Transaction Amount Distribution: Fraud vs Legitimate\n(TRANSFER + CASH_OUT only)',
             fontsize=13, fontweight='bold')

# ---- Top: Linear Scale ----
axes[0].hist(legit_rows, bins=80, color='#3498db', alpha=0.6, label='Legitimate', density=True)
axes[0].hist(fraud_rows,  bins=80, color='#e74c3c', alpha=0.7, label='Fraud',      density=True)
axes[0].set_title('Linear Scale')
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Density')
axes[0].legend()

# Add vertical lines for medians
axes[0].axvline(legit_rows.median(), color='#2980b9', linestyle='--', linewidth=1.5, label=f'Legit Median: {legit_rows.median():,.0f}')
axes[0].axvline(fraud_rows.median(), color='#c0392b', linestyle='--', linewidth=1.5, label=f'Fraud Median: {fraud_rows.median():,.0f}')
axes[0].legend()

# ---- Bottom: Log Scale (better for visualizing wide-range data) ----
# np.log1p applies log(x+1) so we avoid log(0) errors
axes[1].hist(np.log1p(legit_rows), bins=80, color='#3498db', alpha=0.6, label='Legitimate', density=True)
axes[1].hist(np.log1p(fraud_rows),  bins=80, color='#e74c3c', alpha=0.7, label='Fraud',      density=True)
axes[1].set_title('Log Scale  [log(amount + 1)]')
axes[1].set_xlabel('log(Amount + 1)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'amount_distribution.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved to: {save_path}')
print(f'\n  Legit  median amount: {legit_rows.median():,.2f}')
print(f'  Fraud  median amount: {fraud_rows.median():,.2f}')

**What Cell 9 does:**  
Plots how transaction amounts are distributed for fraud vs. legitimate transactions. The log scale version is more useful because the amounts range widely (from tiny to millions). Dashed vertical lines mark the median amounts. Typically, fraud transactions tend to involve higher amounts.

In [ ]:
# ============================================================
# CELL 10 — What % of Fraud Drains Sender Account to Zero?
# ============================================================

print('--- Checking if fraud drains sender account to 0 ---\n')

# Filter only fraud rows
fraud_only = df_filtered[df_filtered['isFraud'] == 1]

# Check how many fraud rows have newbalanceOrig (sender's post-transaction balance) = 0
# This means the fraudster completely drained the account
drained = fraud_only[fraud_only['newbalanceOrig'] == 0]

pct_drained = (len(drained) / len(fraud_only)) * 100

print(f'  Total fraud transactions          : {len(fraud_only):,}')
print(f'  Fraud with newbalanceOrig = 0     : {len(drained):,}')
print(f'  Percentage that drain to zero     : {pct_drained:.2f}%')
print()
print('🔍 Interpretation:')
print(f'  {pct_drained:.1f}% of fraud transactions completely empty the sender\'s account.')
print('  This is a powerful signal — legitimate transactions rarely drain accounts to zero.')
print('  We will engineer features to capture this pattern (error_orig, error_dest).')

# Also check for legit transactions that happen to drain to zero (for comparison)
legit_only = df_filtered[df_filtered['isFraud'] == 0]
legit_drained = legit_only[legit_only['newbalanceOrig'] == 0]
legit_drained_pct = (len(legit_drained) / len(legit_only)) * 100
print(f'\n  (For comparison) Legit txns that drain to zero: {legit_drained_pct:.2f}%')

**What Cell 10 does:**  
Checks a specific domain insight: in fraud transactions, the sender's account is almost always drained to 0 (`newbalanceOrig = 0`). This is because fraudsters typically transfer/cash out everything available. We compare this rate against legitimate transactions to confirm the feature is discriminative.

In [ ]:
# ============================================================
# CELL 11 — Feature Engineering on df_filtered
# ============================================================

print('Engineering new features on df_filtered...\n')

# ---- Feature 1: balance_diff_orig ----
# How much did the SENDER's balance change?
# If this equals 'amount', the transaction looks normal
df_filtered['balance_diff_orig'] = (
    df_filtered['oldbalanceOrg'] - df_filtered['newbalanceOrig']
)

# ---- Feature 2: balance_diff_dest ----
# How much did the RECEIVER's balance change?
# If this equals 'amount', the transaction looks normal
df_filtered['balance_diff_dest'] = (
    df_filtered['newbalanceDest'] - df_filtered['oldbalanceDest']
)

# ---- Feature 3: error_orig ----
# Discrepancy between the transaction amount and the sender's balance change
# Ideally: amount == balance_diff_orig → error_orig = 0
# In fraud: these don't match → error_orig != 0
df_filtered['error_orig'] = (
    df_filtered['amount'] - df_filtered['balance_diff_orig']
)

# ---- Feature 4: error_dest ----
# Same idea for the receiver's side
# In fraud: receiver's balance doesn't increase by the transaction amount
df_filtered['error_dest'] = (
    df_filtered['amount'] - df_filtered['balance_diff_dest']
)

print('✅ New features created:')
new_features = ['balance_diff_orig', 'balance_diff_dest', 'error_orig', 'error_dest']
for feat in new_features:
    print(f'  • {feat}')

print(f'\n  df_filtered now has {df_filtered.shape[1]} columns (was 11 before).')

# Peek at the new features for fraud vs legit rows
print('\n--- Mean values of new features (Legit vs Fraud) ---')
comparison = df_filtered.groupby('isFraud')[new_features].mean().round(2)
print(comparison)

**What Cell 11 does:**  
Creates 4 new columns that capture **balance manipulation** — the core signal of financial fraud. The idea: in a legitimate transaction, the amount should exactly match the balance changes on both sides. When these don't match (non-zero `error_orig` or `error_dest`), it's a red flag. The table at the end shows these engineered features differ significantly between fraud and legit transactions.

In [ ]:
# ============================================================
# CELL 12 — Correlation of Features with isFraud
# ============================================================

print('Calculating correlation of all numeric features with isFraud...\n')

# Select only numeric columns (drop string columns like 'type', 'nameOrig', 'nameDest')
numeric_cols = df_filtered.select_dtypes(include=[np.number]).columns.tolist()

# .corr() calculates the Pearson correlation coefficient between every pair of columns
# We then pick only the row for 'isFraud' — that row shows correlation with our target
correlations = (
    df_filtered[numeric_cols]
    .corr()['isFraud']         # get only the 'isFraud' column of the correlation matrix
    .drop('isFraud')           # remove the self-correlation row (isFraud with isFraud = 1.0)
    .sort_values()             # sort from most negative to most positive correlation
)

print('Correlation with isFraud (sorted):')
print(correlations.round(4).to_string())

# ---- Horizontal Bar Chart ----
fig, ax = plt.subplots(figsize=(10, 7))

# Color bars: blue for negative correlation, red for positive correlation
bar_colors = ['#e74c3c' if v > 0 else '#3498db' for v in correlations.values]

bars = ax.barh(correlations.index, correlations.values, color=bar_colors, edgecolor='black', linewidth=0.5)

ax.set_title('Feature Correlation with isFraud (Target Variable)', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')  # vertical line at 0

# Add value labels to bars
for bar, val in zip(bars, correlations.values):
    ax.text(
        val + (0.005 if val >= 0 else -0.005),
        bar.get_y() + bar.get_height() / 2,
        f'{val:.4f}',
        va='center',
        ha='left' if val >= 0 else 'right',
        fontsize=8
    )

plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'feature_correlation.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Plot saved to: {save_path}')
print(f'\nTop positively correlated feature: {correlations.idxmax()} ({correlations.max():.4f})')
print(f'Top negatively correlated feature: {correlations.idxmin()} ({correlations.min():.4f})')

**What Cell 12 does:**  
Calculates how strongly each numeric feature is correlated with `isFraud` (the target). A value of +1 means perfectly positively correlated (both increase together), −1 means perfectly negatively correlated, and 0 means no linear relationship. This tells us which features are most useful for predicting fraud. Features like `error_orig` and `error_dest` should show high correlation.

In [ ]:
# ============================================================
# CELL 13 — Time Distribution: Fraud vs Legit (by Step/Hour)
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Transaction Volume Over Time (step = hour of simulation)\nFraud vs Legitimate',
             fontsize=13, fontweight='bold')

# ---- Top: Raw counts per step ----
# Group by step and isFraud, then count transactions
# This gives us how many legit and fraud transactions happened each hour
time_data = df_filtered.groupby(['step', 'isFraud']).size().unstack(fill_value=0)
# .unstack() pivots isFraud (0 and 1) into separate columns

time_data.columns = ['Legitimate', 'Fraud']

axes[0].fill_between(time_data.index, time_data['Legitimate'], alpha=0.5, color='#3498db', label='Legitimate')
axes[0].fill_between(time_data.index, time_data['Fraud'],       alpha=0.8, color='#e74c3c', label='Fraud')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('Transaction Count per Hour')
axes[0].legend()

# ---- Bottom: Fraud count only (zoomed in) ----
# Rolling mean smooths out noise — average over 12-hour windows
fraud_rolling = time_data['Fraud'].rolling(window=12, center=True).mean()

axes[1].bar(time_data.index, time_data['Fraud'], color='#e74c3c', alpha=0.5, label='Fraud per hour')
axes[1].plot(time_data.index, fraud_rolling, color='#c0392b', linewidth=2, label='12-hr rolling avg')
axes[1].set_xlabel('Step (Hour of Simulation)')
axes[1].set_ylabel('Fraud Count')
axes[1].set_title('Fraud Transactions Over Time (Zoomed In)')
axes[1].legend()

plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'time_distribution.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved to: {save_path}')

**What Cell 13 does:**  
Plots transaction volume over time (each 'step' = 1 hour of the 30-day simulation). The top chart shows both legitimate and fraud transactions across time. The bottom zooms in on just fraud. A 12-hour rolling average line smooths out hour-to-hour noise to reveal broader patterns. This helps us see if fraud is uniformly distributed or peaks at certain times.

In [ ]:
# ============================================================
# CELL 14 — Full EDA Summary
# ============================================================

# Get top 3 most positively correlated features for the report
top_features = correlations.sort_values(ascending=False).head(3)

print('=' * 65)
print('          EDA SUMMARY — PAYSIM FRAUD DETECTION')
print('=' * 65)

print(f'''
📊 DATASET OVERVIEW
   Total Rows           : {df.shape[0]:,}
   Total Columns        : {df.shape[1]}
   Missing Values       : None (clean dataset)

🎯 TARGET VARIABLE (isFraud)
   Fraud Transactions   : {fraud_count:,}  ({fraud_pct:.4f}%)
   Legit Transactions   : {legit_count:,}  ({legit_pct:.2f}%)
   Imbalance Ratio      : ~{int(legit_count/fraud_count)}:1  (legit:fraud)

🔬 FRAUD PATTERNS FOUND
   • Fraud occurs ONLY in TRANSFER and CASH_OUT types
   • {pct_drained:.1f}% of fraud drains sender account to 0 (newbalanceOrig = 0)
   • Fraud amounts tend to be larger than legitimate amounts
   • No obvious time-of-day pattern for fraud concentration

📐 FEATURE ENGINEERING (4 new features added)
   • balance_diff_orig  : Sender balance change
   • balance_diff_dest  : Receiver balance change  
   • error_orig         : Amount vs sender balance change discrepancy
   • error_dest         : Amount vs receiver balance change discrepancy

🏆 TOP CORRELATED FEATURES (with isFraud)
''')

for feat, val in top_features.items():
    print(f'   • {feat}: {val:.4f}')

print(f'''
⚙️  MODELING IMPLICATIONS
   • Plain accuracy is useless — a dummy model gets 99.87% by predicting all as legit
   • Use: Precision, Recall, F1 Score — especially Recall (catch actual frauds!)
   • Class imbalance handled via class weights, NOT SMOTE (PySpark MLlib compatible)
   • Filter dataset to TRANSFER + CASH_OUT only before training
   
📁 FILTERED DATASET
   Rows in df_filtered : {df_filtered.shape[0]:,}
   Columns             : {df_filtered.shape[1]}
''')
print('=' * 65)

**What Cell 14 does:**  
Prints a clean, structured summary of all findings from the EDA. This is useful for writing the project report and for understanding what the data tells us before building models. It combines all the numbers computed in previous cells into one readable output.

In [ ]:
# ============================================================
# CELL 15 — Export 10,000 Random Rows for Teammate (Member 1)
# ============================================================

print('Exporting 10,000 random rows for Kafka pipeline testing...\n')

# Take a random sample of 10,000 rows from df_filtered
# random_state=42 makes the sample reproducible — same rows every time you run it
sample_10k = df_filtered.sample(n=10_000, random_state=42)

# Confirm fraud is still present in the sample (important for realistic testing)
sample_fraud_count = sample_10k['isFraud'].sum()
print(f'  Sample size     : {len(sample_10k):,} rows')
print(f'  Fraud in sample : {sample_fraud_count} rows ({sample_fraud_count/len(sample_10k)*100:.2f}%)')

# ---- Save as CSV ----
csv_path = 'sample_10k.csv'
sample_10k.to_csv(csv_path, index=False)  # index=False prevents writing row numbers to file
print(f'\n✅ CSV saved  : {csv_path}')

# ---- Save as Parquet ----
# Parquet is a compressed binary format — much faster to read than CSV for large data
# PySpark works very efficiently with Parquet files
parquet_path = 'sample_10k.parquet'
sample_10k.to_parquet(parquet_path, index=False)
print(f'✅ Parquet saved: {parquet_path}')

print(f'''
📦 Files for Member 1 (Kafka pipeline):
   → sample_10k.csv     : Send rows as JSON messages to Kafka topic
   → sample_10k.parquet : Use directly in Spark Structured Streaming tests
''')

**What Cell 15 does:**  
Takes a random 10,000-row sample from the filtered dataset and saves it in two formats — CSV (simple text, easy to read) and Parquet (compressed binary, much faster for Spark). These files are handed off to Member 1 who can use them to test their Kafka streaming pipeline without needing the full 6.3M row dataset.

In [ ]:
# ============================================================
# CELL 16 — Final Feature Schema for Member 1
# ============================================================

print('=' * 65)
print('  FEATURE SCHEMA — What Member 1 must include in Kafka messages')
print('=' * 65)
print()
print('Each Kafka message (JSON) must contain the following fields:')
print()

# Get column names and data types from the filtered dataset
# We exclude columns that will be dropped (nameOrig, nameDest, isFlaggedFraud)
# and include only what the ML model needs

# Define which columns the ML model needs
ml_required_columns = [
    'step', 'type', 'amount',
    'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest',
    'balance_diff_orig', 'balance_diff_dest',
    'error_orig', 'error_dest',
    'isFraud'   # ground truth label — included in sample for testing, omit in live system
]

print(f'  {"Field Name":<22} {"Data Type":<12} {"Description"}')
print(f'  {"-"*22} {"-"*12} {"-"*35}')

# Create a simple description dictionary for each column
descriptions = {
    'step'             : 'Hour of simulation (int)',
    'type'             : 'TRANSFER or CASH_OUT (string)',
    'amount'           : 'Transaction amount (float)',
    'oldbalanceOrg'    : 'Sender balance BEFORE txn (float)',
    'newbalanceOrig'   : 'Sender balance AFTER txn (float)',
    'oldbalanceDest'   : 'Receiver balance BEFORE txn (float)',
    'newbalanceDest'   : 'Receiver balance AFTER txn (float)',
    'balance_diff_orig': 'oldbalanceOrg - newbalanceOrig (float)',
    'balance_diff_dest': 'newbalanceDest - oldbalanceDest (float)',
    'error_orig'       : 'amount - balance_diff_orig (float)',
    'error_dest'       : 'amount - balance_diff_dest (float)',
    'isFraud'          : 'Ground truth label — for testing only'
}

for col in ml_required_columns:
    dtype = str(df_filtered[col].dtype) if col in df_filtered.columns else 'N/A'
    desc  = descriptions.get(col, '')
    print(f'  {col:<22} {dtype:<12} {desc}')

print()
print('⚠️  Note: Do NOT include nameOrig, nameDest, isFlaggedFraud in Kafka messages.')
print('   These columns are dropped before model inference.')
print()
print('✅ EDA Complete! All outputs saved.')
print('   Next step → Run feature_engineering.py (Task 2)')

**What Cell 16 does:**  
Prints a clear schema (field list + data types + descriptions) that Member 1 needs to follow when building their Kafka producer. Each row in the CSV maps to one Kafka message. This ensures both team members are working with the same data structure, making integration seamless.